# Configuring HBR SHASHb parameters

An HBR model is a **likelihood** plus a **prior** on each of its parameters. Here `SHASHb`, with `mu` (mean/location), `sigma` (std/scale), `epsilon` (skew) and
`delta` (kurtosis/tail weight).

What you can configure for HBR:

1. **The four parameters** — how each one reshapes the distribution.
2. **Basis function** — how flexible the curve over age is.
3. **Prior distribution** — the plausible range of each parameter.
4. **Softplus mapping** — keeps parameters positive.


> `mu`'s intercept also can have a **random effect** across batches (sites,
> scanners). Configurable too, but won't be explored in this notebook.

In [8]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from ipywidgets import (interact, interactive_output, Checkbox, Dropdown,
                        FloatSlider, IntSlider, Layout, Tab, VBox)
from IPython.display import display

# Reuse the toolkit's own SHASH math so this playground matches the model.
from pcntoolkit.math_functions.shash import S, S_inv, m1m2
from pcntoolkit import BsplineBasisFunction

warnings.simplefilter("ignore")
%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 4.5)

Helpers: the SHASHb density, its centiles, and the softplus mapping.

In [9]:
CONST2 = -0.5 * np.log(2 * np.pi)  # normal log-normalising constant
AGE = np.linspace(5, 85, 200)


def shashb_pdf(y, mu, sigma, epsilon, delta):
    # Probability density of a SHASHb(mu, sigma, epsilon, delta) variable.
    # m1m2 returns (mean, raw second moment); var = raw_second - mean^2.
    mean, raw_second = m1m2(epsilon, delta)
    true_sigma = np.sqrt(raw_second - mean ** 2)
    # Map y back to the standardised SHASH domain.
    x = ((y - mu) / sigma) * true_sigma + mean
    Sx = S(x, epsilon, delta)
    log_base = (
        CONST2
        + np.log(delta)
        + 0.5 * np.log1p(Sx ** 2)
        - 0.5 * np.log1p(x ** 2)
        - 0.5 * Sx ** 2
    )
    # Change-of-variables Jacobian for the (y -> x) affine map.
    return np.exp(log_base) * true_sigma / sigma


def shashb_centiles(centiles, mu, sigma, epsilon, delta):
    # Quantiles (centiles) of a SHASHb(mu, sigma, epsilon, delta) variable.
    z = norm.ppf(np.asarray(centiles, dtype=float))  # std-normal quantiles
    mean, raw_second = m1m2(epsilon, delta)
    true_sigma = np.sqrt(raw_second - mean ** 2)
    x = S_inv(z, epsilon, delta)                     # base SHASH quantiles
    return ((x - mean) / true_sigma) * sigma + mu


def softplus_mapping(x, a, b, c=0.0):
    # Affine-parameterised softplus, matching Prior.apply_mapping.
    return np.log1p(np.exp((x - a) / b)) * b + c


def make_basis(nknots, degree):
    # Fit the B-spline on the age grid and return the design matrix.
    bf = BsplineBasisFunction(basis_column=0, nknots=nknots, degree=degree)
    x = AGE.reshape(-1, 1)
    bf.fit(x)
    return np.asarray(bf.transform(x))


# Sanity check: the density must integrate to ~1.
_grid = np.linspace(-15, 15, 20001)
_area = np.trapz(shashb_pdf(_grid, 0.0, 1.0, 0.8, 0.9), _grid)
print(f"pdf integrates to {_area:.4f}")

pdf integrates to 1.0000


## 1. How the parameters affect the distribution

Every HBR fit is this shape, evaluated at each age.

- `mu` — location. Slides the distribution left and right.
- `sigma` — scale. Stretches it wider or narrower.
- `epsilon` — skew. `0` is symmetric, positive leans right.
- `delta` — tail weight. `1` is normal-like, `< 1` heavier, `> 1` lighter.

In [10]:
@interact(
    mu=FloatSlider(value=0.0, min=-3.0, max=3.0, step=0.1, description="mu"),
    sigma=FloatSlider(value=1.0, min=0.3, max=3.0, step=0.1,
                      description="sigma"),
    epsilon=FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.1,
                        description="epsilon"),
    delta=FloatSlider(value=1.0, min=0.3, max=2.5, step=0.05,
                      description="delta"),
)
def _plot_shape(mu, sigma, epsilon, delta):
    y = np.linspace(-8, 8, 1000)
    pdf = shashb_pdf(y, mu, sigma, epsilon, delta)
    # A reference Normal with the same mu, sigma.
    ref = norm.pdf(y, mu, sigma)

    fig, ax = plt.subplots()
    ax.plot(y, pdf, lw=2.5, label="SHASHb")
    ax.plot(y, ref, lw=1.5, ls=":", color="grey", label="Normal(mu, sigma)")
    for c, ls in zip([0.05, 0.5, 0.95], ["--", "-", "--"]):
        q = shashb_centiles([c], mu, sigma, epsilon, delta)[0]
        ax.axvline(q, color="crimson", ls=ls, lw=1, alpha=0.7)
    ax.set_xlim(-8, 8)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("y")
    ax.set_ylabel("density")
    ax.set_title("SHASHb density (red lines = 5th/50th/95th centiles)")
    ax.legend(loc="upper right")
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='mu', max=3.0, min=-3.0), FloatSlider(value=1.0, desc…

## 2. Basis function

Expands age into smooth columns in a design matrix; `mu(age)` is a weighted sum of them. The first column of the design matrix is the raw covariate, so the fit can always fall back to a straight line. Number of columns = `degree + nknots`

- `nknots` — more knots, more wiggle.
- `degree` — higher degree, smoother each bend.

In the figure:
The left plot draws one line per column. 

The one line going to ~85 is the raw age column, not a spline. This line makes the right plot go to large values (+-100) and always be a straight lines even when we increase nknots and degree

The right plot always has 8 lines because the loop is hardcoded to 8.

In [15]:
@interact(
    nknots=IntSlider(value=5, min=3, max=12, step=1, description="nknots"),
    degree=IntSlider(value=3, min=1, max=5, step=1, description="degree"),
)
def _plot_basis(nknots, degree):
    phi = make_basis(nknots, degree)          # (n_age, k)
    rng = np.random.default_rng(0)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    axes[0].plot(AGE, phi)
    axes[0].set_title(f"B-spline basis ({phi.shape[1]} columns)")
    axes[0].set_xlabel("age")
    # Unit-scale weights: the shapes come from the basis, not the prior.
    for _ in range(8):
        w = rng.normal(0.0, 1.0, phi.shape[1])
        axes[1].plot(AGE, phi @ w, alpha=0.7)
    axes[1].set_title("mu(age) curves the basis can express")
    axes[1].set_xlabel("age")
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=5, description='nknots', max=12, min=3), IntSlider(value=3, description=…

## 3. Prior distribution

Sets which values of a parameter are plausible before seeing data — including
the basis weights, whose prior width decides how wiggly `mu(age)` really gets (eg Narrow prior -> weights shrink to zero -> mu(age) is a flat horizontal line, no matter how many knots.).

- `dist_name` — the family. Only `Normal`, `HalfNormal`, `LogNormal`,
  `Gamma`, `Uniform` exist, spelled exactly like that.
- `dist_params` — a positional tuple, meaning depends on the family.
  eg `Gamma` is `(alpha, beta)`, shape-**rate**, not shape-scale.

> If i want a parameter to be positive i should select the right dist_name (eg LogNormal) or apply mapping (see next section).

In [12]:
def sample_prior(dist_name, p0, p1, n=200000, seed=0):
    rng = np.random.default_rng(seed)
    if dist_name == "Normal":
        return rng.normal(p0, p1, n)
    if dist_name == "HalfNormal":
        return np.abs(rng.normal(0.0, p1, n))
    if dist_name == "LogNormal":
        return rng.lognormal(p0, p1, n)
    if dist_name == "Gamma":          # (alpha, beta) shape-rate, PyMC order
        return rng.gamma(max(p0, 1e-3), 1.0 / max(p1, 1e-3), n)
    if dist_name == "Uniform":
        lo, hi = min(p0, p1), max(p0, p1)
        return rng.uniform(lo, hi + 1e-9, n)
    raise ValueError(dist_name)


@interact(
    dist_name=Dropdown(
        options=["Normal", "HalfNormal", "LogNormal", "Gamma", "Uniform"],
        value="Normal", description="dist"),
    p0=FloatSlider(value=0.0, min=-3.0, max=5.0, step=0.1,
                   description="param 0"),
    p1=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                   description="param 1"),
)
def _plot_prior(dist_name, p0, p1):
    samples = sample_prior(dist_name, p0, p1)
    fig, ax = plt.subplots()
    ax.hist(samples, bins=200, density=True, color="steelblue", alpha=0.8)
    ax.set_xlim(np.percentile(samples, 0.5), np.percentile(samples, 99.5))
    ax.set_xlabel("parameter value")
    ax.set_ylabel("prior density")
    ax.set_title(f"{dist_name}(p0={p0}, p1={p1}) prior")
    plt.show()

interactive(children=(Dropdown(description='dist', options=('Normal', 'HalfNormal', 'LogNormal', 'Gamma', 'Uni…

## 4. Softplus mapping

Pushes an unconstrained sample into the positive domain.

```
f_abc(x) = softplus((x - a) / b) * b + c 

where:
f_abc(x) = mu or sigma or epsilon or delta AFTER TRANSFORMATION (always positive)
x=mu or sigma or epsilon or delta BEFORE TRANSFORMATION
```

- `a` — horizontal shift.
- `b` — scale; larger is gentler, closer to a straight line.
- `c` — vertical shift, a floor on the output.

> `c` only applies if you pass a **3-tuple**. `mapping_params=(0.0, 3.0)`
> leaves the floor at 0.

In [13]:
@interact(
    a=FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.1,
                  description="a (shift)"),
    b=FloatSlider(value=3.0, min=0.5, max=6.0, step=0.1,
                  description="b (scale)"),
    c=FloatSlider(value=0.0, min=0.0, max=2.0, step=0.1,
                  description="c (floor)"),
)
def _plot_softplus(a, b, c):
    x = np.linspace(-8, 8, 400)
    fig, ax = plt.subplots()
    ax.plot(x, softplus_mapping(x, 0.0, 1.0, 0.0), ls=":", color="grey",
            label="softplus (a=0, b=1, c=0)")
    ax.plot(x, softplus_mapping(x, a, b, c), lw=2.5,
            label=f"softplus (a={a}, b={b}, c={c})")
    ax.axhline(c, color="crimson", ls="--", lw=1, alpha=0.7,
               label=f"floor c={c}")
    ax.set_xlabel("unconstrained sample x")
    ax.set_ylabel("mapped (positive) value")
    ax.set_title("Effect of the softplus mapping parameters")
    ax.legend(loc="upper left")
    plt.show()

interactive(children=(FloatSlider(value=0.0, description='a (shift)', max=4.0, min=-4.0), FloatSlider(value=3.…

## 5. Build your configuration

Tune each parameter on its own tab until the centile fan looks plausible, then
copy the code printed underneath.

- `slope mu` / `slope sd` — prior on the basis weights: the shape of the curve.
- `icpt mu` / `icpt sd` — prior on the intercept: its overall level.
- `softplus` — tick to constrain the parameter positive; `a`, `b`, `c` then
  become the `mapping_params`.

> `sigma` and `delta` must stay positive (that holds for every HBR SHASHb we define!!**), so they ship with `softplus` on.
> Untick it and the emitted code will raise when you fit.


** `sigma` must be negative because it is a width (from its definition) and `delta` must be positive because  the SHASHb density takes `log(delta)`.

Two ways to guarantee it:

1. `Normal` prior + `mapping="softplus"` — sample freely, squash to positive.
   Preferred: the sampler works better in an unconstrained space.
2. A prior that is already positive (`HalfNormal`, `LogNormal`, `Gamma`),
   no mapping needed.

If `sigma` is configured to depend on age (`linear=True` + a basis function).Softplus (1.) on the output handles
this automatically. Prior (2.) does not: a linear combination of positive weights
can still dip negative, so a positive prior on the weights is not enough.


In [14]:
# --- code generation -------------------------------------------------------

def fmt_mapping(use_softplus, a, b, c):
    # Mapping kwargs for one prior. With softplus off the prior uses the
    # default `identity` mapping, which ignores a and b but still ADDS c --
    # so emit nothing at all rather than a misleading tuple.
    if not use_softplus:
        return ""
    params = f"({a}, {b})" if c == 0.0 else f"({a}, {b}, {c})"
    return f'\n    mapping="softplus",\n    mapping_params={params},'


def build_config(nknots, degree,
                 mu_slope_mu, mu_slope_sd, mu_icpt_mu, mu_icpt_sd,
                 mu_sp, mu_a, mu_b, mu_c,
                 sig_slope_mu, sig_slope_sd, sig_icpt_mu, sig_icpt_sd,
                 sig_sp, sig_a, sig_b, sig_c,
                 eps_mu, eps_sd, eps_sp, eps_a, eps_b, eps_c,
                 dlt_mu, dlt_sd, dlt_sp, dlt_a, dlt_b, dlt_c):
    # The random effect on mu's intercept is fixed, so it is written verbatim.
    basis = (f"BsplineBasisFunction(basis_column=0, nknots={nknots}, "
             f"degree={degree})")
    return f'''mu = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal",
                     dist_params=({mu_slope_mu}, {mu_slope_sd})),
    intercept=make_prior(
        random=True,
        mu=make_prior(dist_name="Normal",
                      dist_params=({mu_icpt_mu}, {mu_icpt_sd})),
        sigma=make_prior(dist_name="Normal", dist_params=(1.0, 1.0),
                         mapping="softplus", mapping_params=(0.0, 3.0)),
    ),
    basis_function={basis},{fmt_mapping(mu_sp, mu_a, mu_b, mu_c)}
)
sigma = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal",
                     dist_params=({sig_slope_mu}, {sig_slope_sd})),
    intercept=make_prior(dist_name="Normal",
                         dist_params=({sig_icpt_mu}, {sig_icpt_sd})),
    basis_function={basis},{fmt_mapping(sig_sp, sig_a, sig_b, sig_c)}
)
epsilon = make_prior(
    dist_name="Normal",
    dist_params=({eps_mu}, {eps_sd}),{fmt_mapping(eps_sp, eps_a, eps_b, eps_c)}
)
delta = make_prior(
    dist_name="Normal",
    dist_params=({dlt_mu}, {dlt_sd}),{fmt_mapping(dlt_sp, dlt_a, dlt_b, dlt_c)}
)
likelihood = SHASHbLikelihood(mu, sigma, epsilon, delta)'''


def apply_map(x, use_softplus, a, b, c):
    # Mirror BasePrior.apply_mapping: identity ignores a and b, but adds c.
    if not use_softplus:
        return x + c
    return softplus_mapping(x, a, b, c)


# --- widgets, one tab per parameter ----------------------------------------

def _sl(v, lo, hi, st, desc):
    return FloatSlider(value=v, min=lo, max=hi, step=st, description=desc,
                       layout=Layout(width="320px"))


def _map_row(on, a, b, c):
    # softplus toggle + its three mapping params.
    return (Checkbox(value=on, description="softplus", indent=False),
            _sl(a, -4.0, 4.0, 0.1, "map a"),
            _sl(b, 0.5, 6.0, 0.1, "map b"),
            _sl(c, 0.0, 1.5, 0.05, "map c"))


W = {}
W["nknots"] = IntSlider(value=5, min=3, max=10, step=1, description="nknots")
W["degree"] = IntSlider(value=3, min=1, max=5, step=1, description="degree")
W["seed"] = IntSlider(value=0, min=0, max=20, step=1, description="seed")

W["mu_slope_mu"] = _sl(0.0, -5.0, 5.0, 0.1, "slope mu")
W["mu_slope_sd"] = _sl(3.0, 0.5, 20.0, 0.5, "slope sd")
W["mu_icpt_mu"] = _sl(0.0, -5.0, 5.0, 0.1, "icpt mu")
W["mu_icpt_sd"] = _sl(1.0, 0.1, 5.0, 0.1, "icpt sd")
W["mu_sp"], W["mu_a"], W["mu_b"], W["mu_c"] = _map_row(False, 0.0, 1.0, 0.0)

W["sig_slope_mu"] = _sl(0.0, -5.0, 5.0, 0.1, "slope mu")
W["sig_slope_sd"] = _sl(2.0, 0.5, 6.0, 0.5, "slope sd")
W["sig_icpt_mu"] = _sl(1.0, -2.0, 3.0, 0.1, "icpt mu")
W["sig_icpt_sd"] = _sl(1.0, 0.1, 4.0, 0.1, "icpt sd")
W["sig_sp"], W["sig_a"], W["sig_b"], W["sig_c"] = _map_row(True, 0.0, 2.0, 0.0)

W["eps_mu"] = _sl(0.0, -1.5, 1.5, 0.1, "mu")
W["eps_sd"] = _sl(1.0, 0.1, 3.0, 0.1, "sd")
W["eps_sp"], W["eps_a"], W["eps_b"], W["eps_c"] = _map_row(
    False, 0.0, 1.0, 0.0)

W["dlt_mu"] = _sl(1.0, -1.0, 3.0, 0.1, "mu")
W["dlt_sd"] = _sl(1.0, 0.1, 3.0, 0.1, "sd")
W["dlt_sp"], W["dlt_a"], W["dlt_b"], W["dlt_c"] = _map_row(True, 0.0, 3.0, 0.6)


def _tab(keys):
    return VBox([W[k] for k in keys])


tabs = Tab(children=[
    _tab(["mu_slope_mu", "mu_slope_sd", "mu_icpt_mu", "mu_icpt_sd",
          "mu_sp", "mu_a", "mu_b", "mu_c"]),
    _tab(["sig_slope_mu", "sig_slope_sd", "sig_icpt_mu", "sig_icpt_sd",
          "sig_sp", "sig_a", "sig_b", "sig_c"]),
    _tab(["eps_mu", "eps_sd", "eps_sp", "eps_a", "eps_b", "eps_c"]),
    _tab(["dlt_mu", "dlt_sd", "dlt_sp", "dlt_a", "dlt_b", "dlt_c"]),
    _tab(["nknots", "degree", "seed"]),
])
for i, t in enumerate(["mu", "sigma", "epsilon", "delta", "basis"]):
    tabs.set_title(i, t)


# --- plot + emit -----------------------------------------------------------

def _render(nknots, degree, seed,
            mu_slope_mu, mu_slope_sd, mu_icpt_mu, mu_icpt_sd,
            mu_sp, mu_a, mu_b, mu_c,
            sig_slope_mu, sig_slope_sd, sig_icpt_mu, sig_icpt_sd,
            sig_sp, sig_a, sig_b, sig_c,
            eps_mu, eps_sd, eps_sp, eps_a, eps_b, eps_c,
            dlt_mu, dlt_sd, dlt_sp, dlt_a, dlt_b, dlt_c):
    phi = make_basis(nknots, degree)
    rng = np.random.default_rng(seed)

    # mu(age): basis weights plus a scalar intercept draw, then its mapping.
    w_mu = rng.normal(mu_slope_mu, mu_slope_sd, phi.shape[1])
    raw_mu = phi @ w_mu + rng.normal(mu_icpt_mu, mu_icpt_sd)
    mu_age = apply_map(raw_mu, mu_sp, mu_a, mu_b, mu_c)

    # sigma(age): linear in the basis, then softplus to keep it positive.
    w_sig = rng.normal(sig_slope_mu, sig_slope_sd, phi.shape[1])
    raw_sig = phi @ w_sig + rng.normal(sig_icpt_mu, sig_icpt_sd)
    sigma_age = apply_map(raw_sig, sig_sp, sig_a, sig_b, sig_c)

    # epsilon and delta are scalar priors: one draw each, not per-age.
    epsilon = apply_map(rng.normal(eps_mu, eps_sd),
                        eps_sp, eps_a, eps_b, eps_c)
    delta = apply_map(rng.normal(dlt_mu, dlt_sd), dlt_sp, dlt_a, dlt_b, dlt_c)

    # A non-positive sigma or delta is exactly what the softplus prevents.
    if np.any(sigma_age <= 0) or delta <= 0:
        print("sigma or delta went non-positive -- turn softplus back on.")
        return

    cents = [0.05, 0.25, 0.5, 0.75, 0.95]
    curves = np.array([
        [shashb_centiles([c], m, s, epsilon, delta)[0] for c in cents]
        for m, s in zip(mu_age, sigma_age)
    ])  # (n_age, n_cent)

    fig, ax = plt.subplots()
    ax.fill_between(AGE, curves[:, 0], curves[:, -1], alpha=0.2,
                    color="steelblue", label="5th-95th centile")
    ax.fill_between(AGE, curves[:, 1], curves[:, -2], alpha=0.3,
                    color="steelblue", label="25th-75th centile")
    ax.plot(AGE, curves[:, 2], color="navy", lw=2, label="median")
    ax.set_xlabel("age")
    ax.set_ylabel("response")
    ax.set_title(f"Prior-predictive centile fan "
                 f"(epsilon={epsilon:.2f}, delta={delta:.2f})")
    ax.legend(loc="upper left")
    plt.show()

    print(build_config(nknots, degree,
                       mu_slope_mu, mu_slope_sd, mu_icpt_mu, mu_icpt_sd,
                       mu_sp, mu_a, mu_b, mu_c,
                       sig_slope_mu, sig_slope_sd, sig_icpt_mu, sig_icpt_sd,
                       sig_sp, sig_a, sig_b, sig_c,
                       eps_mu, eps_sd, eps_sp, eps_a, eps_b, eps_c,
                       dlt_mu, dlt_sd, dlt_sp, dlt_a, dlt_b, dlt_c))


out = interactive_output(_render, W)
display(tabs, out)

Output()

### Analysis of the plot

mu and std tab:

> the slope and intercept std and  are just a number. How this numbers affects the centiles is random as it depends on a random draw:

> parameter (eg mu) = mean + std * z, where z is a random draw from a normal distr. 

> In our example above we draw from ONE specific draw. What std genuinely controls is spread across draws, which one draw here cannot show.

> As a result i would say the most important is to set up a good slope and intercept mu and then think how much this mu should be spread. 


mu tab
> when mu slope mu is near zero, the only thing left in the weights is the random part, so the curve bends a lot

basis tab:
> Some `nknots`/`degree` combinations make the centile band look like it
> vanishes. Nothing is broken. Changing the number of basis columns changes how
> many random numbers are drawn, which re-rolls every parameter (including sigma) further
> down the same seed — you get a different draw, not a different model. Nudge
> the `seed` slider and the band comes back (that is also what the fit() code would do automatically).
>
> And it happens in this draw that the band is small relative to the y-axis (mean)
 